# Lesson 4.5: Practice Friday Memo Final

You'll combine everything from Lessons 4.1 through 4.5 to produce the deliverable Hannah needs at 2 PM.


## Artifact 1: A cleaned, enriched customer Dataframe

## Step 1. Set ups and load CSV

In [1]:
# Import libraries 
import pandas as pd
import numpy as np

# Set options 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load CSV file
df = pd.read_csv(
    'finflow-customers-200k.csv',
    # change datatype to Date
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

# Check if data loaded properly
print(f"Customer data: {df.shape}")
df.head(2)

Customer data: (200000, 16)


,customer_id,first_name,last_name,email,city,state,country,signup_date,plan_tier,monthly_fee,last_login_date,total_sessions,avg_session_duration_minutes,support_tickets_opened,active,churn_date
0,FF-00001,Danielle,Johnson,danielle.johnson1@example.com,Johnberg,WA,US,2025-10-17,Basic,9.99,2025-12-25,134,4.70,0,True,NaT
1,FF-00002,Curtis,Yang,curtis.yang2@example.com,New Roberttown,CO,US,2025-11-24,Pro,29.99,2025-12-14,236,3.40,0,True,NaT


## Inspecting the marketing team's plan metadata

Load it. Look at it. Document every problem before fixing anything.


In [2]:
# Load other CSV file 
plans_raw = pd.read_csv(
    'plan_metadata_raw.csv',
    # Change dataype of lauch_date to Date
    parse_dates=['launch_date']
)

# Check data
print(f"Shape: {plans_raw.shape}")
print()
plans_raw

Shape: (5, 5)



,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date
0,Basic,Starter Plan,Individual,9.99,2019-03-15
1,PRO,Professional,SMB,29.99,2020-08-01
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30
3,PRO,Professional,SMB,29.99,2020-08-01
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12


In [3]:
# Survey the problems
print("Unique plan_tier values (as-is):")
# Check if there are white space in text
for value in plans_raw['plan_tier'].unique():
    print(f"  '{value}' (length {len(value)})")
print()

# Missing values
print("Missing values per column:")
print(plans_raw.isna().sum())
print()

# Duplicates
print(f"Duplicate rows: {plans_raw.duplicated().sum()}")

Unique plan_tier values (as-is):
  ' Basic ' (length 7)
  'PRO' (length 3)
  ' enterprise ' (length 12)
  'Legacy' (length 6)

Missing values per column:
plan_tier             0
marketing_name        0
target_segment        0
monthly_fee_listed    1
launch_date           0
dtype: int64

Duplicate rows: 1


**Observation:** Five rows total, four distinct after deduplication.
Plan tier values have whitespace and capitalization mess that will prevent
the join column from matching the customer data. monthly_fee_listed is
missing for Legacy. Need to clean before merging.


In [4]:
# Strip whitespace, then title-case
plans_raw['plan_tier_clean'] = plans_raw['plan_tier'].str.strip().str.title()

# Check if whitespace removed and title-cased
print("Cleaned plan_tier values:")
for value in plans_raw['plan_tier_clean'].unique():
    print(f"  '{value}' (length {len(value)})")

Cleaned plan_tier values:
  'Basic' (length 5)
  'Pro' (length 3)
  'Enterprise' (length 10)
  'Legacy' (length 6)


In [5]:
# How many rows before
print(f"Rows before dedupe: {len(plans_raw)}")

# Drop duplicates on the cleaned column
plans = plans_raw.drop_duplicates(subset=['plan_tier_clean'], keep='first').copy()

print(f"Rows after dedupe: {len(plans)}")
plans

Rows before dedupe: 5
Rows after dedupe: 4


,plan_tier,marketing_name,target_segment,monthly_fee_listed,launch_date,plan_tier_clean
0,Basic,Starter Plan,Individual,9.99,2019-03-15,Basic
1,PRO,Professional,SMB,29.99,2020-08-01,Pro
2,enterprise,Enterprise Plus,Enterprise,199.00,2021-11-30,Enterprise
4,Legacy,Heritage Plan,Individual,NaN,2017-06-12,Legacy


In [6]:
# Confirm the NaN remains
print("monthly_fee_listed values:")
print(plans[['plan_tier_clean', 'monthly_fee_listed']])

monthly_fee_listed values:
  plan_tier_clean  monthly_fee_listed
0           Basic                9.99
1             Pro               29.99
2      Enterprise              199.00
4          Legacy                 NaN


**Observation:** Leaving the Legacy monthly_fee_listed as NaN rather than
filling. The customer data already has a per-customer monthly_fee, so the
listed price isn't critical for the join. If Hannah asks during the
readout, we can investigate.

In [7]:
print("Customer plan_tier values:")
print(df['plan_tier'].value_counts())

Customer plan_tier values:
plan_tier
Basic         119881
Pro            60229
Enterprise     19890
Name: count, dtype: int64


In [8]:
# Inner join: only rows where plan_tier_clean matches on both sides
df_inner = pd.merge(
    df,
    plans,
    left_on='plan_tier',
    right_on='plan_tier_clean',
    how='inner'
)

# Print Results after inner join
print(f"Customer rows before merge: {len(df)}")
print(f"Customer rows after inner merge: {len(df_inner)}")
print(f"Columns added: {df_inner.shape[1] - df.shape[1]}")
print()
print("Tier counts in inner merge result:")
print(df_inner['plan_tier_clean'].value_counts())

Customer rows before merge: 200000
Customer rows after inner merge: 200000
Columns added: 6

Tier counts in inner merge result:
plan_tier_clean
Basic         119881
Pro            60229
Enterprise     19890
Name: count, dtype: int64


In [9]:
# Left join table based on plan_tier column
df_enriched = pd.merge(
    df,
    plans,
    left_on='plan_tier',
    right_on='plan_tier_clean',
    how='left'
)

#Check results
print(f"Shape after left merge: {df_enriched.shape}")
print(f"NaN counts in metadata columns:")
print(df_enriched[['marketing_name', 'target_segment', 'monthly_fee_listed', 'launch_date']].isna().sum())

Shape after left merge: (200000, 22)
NaN counts in metadata columns:
marketing_name        0
target_segment        0
monthly_fee_listed    0
launch_date           0
dtype: int64


## Export df_enriched to CSV

In [10]:
# Saved the table as a CSV
df_enriched.to_csv('finflow-enriched-200k.csv', index=False)
print("Saved finflow-enriched-200k.csv")

Saved finflow-enriched-200k.csv


## Artifact 2: The consolidated per-tier memo table.
Build a DataFrame with one row per marketing_name (Starter Plan, Professional, Enterprise Plus) and these columns:

target_segment

customer_count

total_mrr (sum of monthly_fee from active customers only)

churn_rate_pct

avg_sessions_active (mean total_sessions among active customers)

avg_sessions_churned (mean total_sessions among churned customers)

mrr_pct_of_total (active MRR as percent of total active MRR across all tiers)

Save as friday-memo-summary-final.csv.

In [11]:
# Split data into groups to reuse 
active = df_enriched[df_enriched['active']]
churned = df_enriched[~df_enriched['active']]
by_marketing_name = df_enriched.groupby(['marketing_name', 'target_segment'])

# Compute each column as a separate Series indexed by plan_tier
customer_count = by_marketing_name.size()
total_mrr = active.groupby(['marketing_name', 'target_segment'])['monthly_fee'].sum()
churn_rate_pct = (1 - by_marketing_name['active'].mean()) * 100
avg_sessions_active = active.groupby(['marketing_name', 'target_segment'])['total_sessions'].mean()
avg_sessions_churned = churned.groupby(['marketing_name', 'target_segment'])['total_sessions'].mean()
mrr_pct_of_total = (active.groupby(['marketing_name', 'target_segment'])['monthly_fee'].sum() / active['monthly_fee'].sum()) * 100

# Combine the six Series side by side: one row per plan, one column per metric
memo_summary = pd.DataFrame({
    'customer_count': customer_count,
    'total_mrr': total_mrr,
    'churn_rate_pct': churn_rate_pct,
    'avg_sessions_active': avg_sessions_active,
    'avg_session_churned': avg_sessions_churned,
    'mrr_pct_of_total': mrr_pct_of_total
})

# Round each column so the table is easy to read
memo_summary = memo_summary.round({
    'total_mrr': 2,             
    'churn_rate_pct': 1,         
    'avg_sessions_active': 2,    
    'avg_session_churned': 2,
    'mrr_pct_of_total': 1,
})

# Print results
memo_summary

,,customer_count,total_mrr,churn_rate_pct,avg_sessions_active,avg_session_churned,mrr_pct_of_total
marketing_name,target_segment,,,,,,
Enterprise Plus,Enterprise,19890,3171264.00,19.90,210.15,25.58,56.90
Professional,SMB,60229,1443388.71,20.10,210.41,25.51,25.90
Starter Plan,Individual,119881,958880.16,19.90,209.48,25.51,17.20


In [12]:
# Save the table 
memo_summary.to_csv('friday-memo-summary-final.csv')
print("Saved friday-memo-summary.csv")

Saved friday-memo-summary.csv


## Artifact 3: A "Legacy" callout

In [13]:
# Outer Join two tables
df_outer = pd.merge(
    df[['customer_id', 'plan_tier']], # Narrowed for readability
    plans,
    left_on='plan_tier',
    right_on='plan_tier_clean',
    how='outer'
)

print(f"Outer merge shape: {df_outer.shape}")
print()
print("Rows where customer_id is NaN (these are unmatched metadata rows):")
print(df_outer[df_outer['customer_id'].isna()])

Outer merge shape: (200001, 8)

Rows where customer_id is NaN (these are unmatched metadata rows):
       customer_id plan_tier_x plan_tier_y marketing_name target_segment  \
139771         NaN         NaN      Legacy  Heritage Plan     Individual   

        monthly_fee_listed launch_date plan_tier_clean  
139771                 NaN  2017-06-12          Legacy  


## Call out
The Legacy tier has no current customers in the 200k dataset. The legacy only appears because of outer join, where there is no customers with 'legacy' plan tier.
This is likely a low priority since there are no current customers with this tier.